Imports


In [1]:
import pandas as pd
import gensim
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

Load Data

In [2]:
df = pd.read_csv('Data/combined_timeline_data.csv')
df.head()

,Date,Source,Summary,Associated Link Title,Associated Link URL,Theme
0,"January 4, 2020",WHO Announces Pneumonia Cases of Unknown Cause,The World Health Organization (WHO) announces ...,WHO on Twitter,https://fraser.stlouisfed.org/docs/historical/...,Pandemic
1,"January 8, 2020",CDC Issues Health Advisory,The Centers for Disease Control and Prevention...,CDC Health Advisory,https://fraser.stlouisfed.org/archival-collect...,Pandemic
2,"January 9, 2020",CDC Notes Appearance of Novel Coronavirus Outb...,The CDC notes the appearance of a novel corona...,CDC Report,https://fraser.stlouisfed.org/archival-collect...,Pandemic
3,"January 17, 2020, 2:00 pm EST",CDC Announces Enhanced Screenings for Those Tr...,The CDC and the Department of Homeland Securit...,CDC Press Release,https://fraser.stlouisfed.org/archival-collect...,Pandemic
4,"January 21, 2020",Washington State Department of Health Announce...,The Washington State Department of Health anno...,Snohomish Health District Media Release,https://fraser.stlouisfed.org/title/state-mate...,Pandemic


Preprocess Data

In [4]:
docs = df["Summary"].tolist()
# Extract only the date part (remove everything after the year)
timestamps = pd.to_datetime(df["Date"].str.extract(r'([A-Za-z]+\s+\d{1,2},\s+\d{4})')[0]).dt.date

# 1. Preprocessing (Required for LDA)
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_lda(text):
    text = re.sub(r'\s+', ' ', text).lower() # Remove extra whitespace & lowercase
    tokens = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words and len(w) > 3]
    return tokens

# Process your existing 'docs' list
processed_docs = [preprocess_lda(doc) for doc in docs]

# 2. Create Dictionary and Corpus
id2word = corpora.Dictionary(processed_docs)
# Filter out words that appear in less than 5 docs or more than 50% of docs
id2word.filter_extremes(no_below=5, no_above=0.5) 
corpus = [id2word.doc2bow(text) for text in processed_docs]

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gianf\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\gianf\AppData\Roaming\nltk_data...


Train LDA Model

In [5]:
# 3. Train LDA Model
# Note: You MUST pre-define the number of topics (k)
num_topics = 12 
lda_model = LdaModel(corpus=corpus, id2word=id2word, num_topics=num_topics, 
                     random_state=42, passes=10, alpha='auto')

View Topics

In [6]:
# 4. View Topics
for idx, topic in lda_model.print_topics(-1):
    print(f"Topic: {idx} \nWords: {topic}")

Topic: 0 
Words: 0.038*"federal" + 0.032*"market" + 0.031*"liquidity" + 0.028*"facility" + 0.028*"reserve" + 0.024*"fund" + 0.022*"commercial" + 0.022*"money" + 0.019*"paper" + 0.018*"rule"
Topic: 1 
Words: 0.029*"temporary" + 0.029*"federal" + 0.029*"reserve" + 0.027*"u.s." + 0.025*"announces" + 0.023*"facility" + 0.021*"bank" + 0.018*"guarantee" + 0.018*"extension" + 0.016*"central"
Topic: 2 
Words: 0.030*"bank" + 0.029*"federal" + 0.021*"reserve" + 0.018*"loan" + 0.017*"commission" + 0.017*"release" + 0.016*"financial" + 0.015*"would" + 0.014*"economic" + 0.014*"banking"
Topic: 3 
Words: 0.056*"federal" + 0.045*"credit" + 0.037*"reserve" + 0.031*"rate" + 0.029*"loan" + 0.028*"board" + 0.021*"primary" + 0.019*"percent." + 0.017*"basis" + 0.017*"point"
Topic: 4 
Words: 0.042*"quarter" + 0.029*"institution" + 0.027*"bank" + 0.026*"asset" + 0.025*"capital" + 0.024*"billion" + 0.024*"announces" + 0.023*"fdic" + 0.019*"board" + 0.018*"policy"
Topic: 5 
Words: 0.091*"u.s." + 0.073*"purchas

Preparing validation

In [7]:
# LDA (Gensim)
lda_topics = [
    [word for word, _ in lda_model.show_topic(t, topn=10)] 
    for t in range(lda_model.num_topics)
]

In [9]:
from gensim.models.coherencemodel import CoherenceModel

def calculate_metrics(topics, corpus_docs, dictionary):
    # 1. Coherence (Cv)
    cm = CoherenceModel(topics=topics, texts=corpus_docs, dictionary=dictionary, coherence='c_v')
    coherence = cm.get_coherence()
    
    # 2. Topic Diversity
    all_words = [word for topic in topics for word in topic]
    unique_words = set(all_words)
    diversity = len(unique_words) / len(all_words)
    
    return coherence, diversity

# Example usage for LDA
# (Repeat for bertopic_topics and top2vec_topics)
cv_lda, div_lda = calculate_metrics(lda_topics, processed_docs, id2word)
print(f"LDA Coherence (C_v): {cv_lda:.4f}")
print(f"LDA Topic Diversity: {div_lda:.4f}")

LDA Coherence (C_v): 0.5557
LDA Topic Diversity: 0.6083
